In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

#%pip install scikit-learn requests
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, classification_report

In [12]:
#%pip install openmeteo-requests
#%pip install requests-cache retry-requests numpy pandas
import openmeteo_requests

import requests_cache
from retry_requests import retry
# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://api.open-meteo.com/v1/forecast"
params = {
	"latitude": 55.9521,
	"longitude": -3.1965,
	"daily": ["weather_code", "temperature_2m_max", "temperature_2m_min", "wind_gusts_10m_max", "rain_sum"],
	"models": "ukmo_seamless",
	"timezone": "Europe/London",
	"past_days": 92,
}
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone: {response.Timezone()}{response.TimezoneAbbreviation()}")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process daily data. The order of variables needs to be the same as requested.
daily = response.Daily()
daily_weather_code = daily.Variables(0).ValuesAsNumpy()
daily_temperature_2m_max = daily.Variables(1).ValuesAsNumpy()
daily_temperature_2m_min = daily.Variables(2).ValuesAsNumpy()
daily_wind_gusts_10m_max = daily.Variables(3).ValuesAsNumpy()
daily_rain_sum = daily.Variables(4).ValuesAsNumpy()

daily_data = {
	"date": pd.date_range(
		start = pd.to_datetime(daily.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(daily.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = daily.Interval()),
		inclusive = "left"
	).tz_convert(response.Timezone().decode())
}

daily_data["weather_code"] = daily_weather_code
daily_data["temperature_2m_max"] = daily_temperature_2m_max
daily_data["temperature_2m_min"] = daily_temperature_2m_min
daily_data["wind_gusts_10m_max"] = daily_wind_gusts_10m_max
daily_data["rain_sum"] = daily_rain_sum

daily_dataframe = pd.DataFrame(data = daily_data)
print("\nDaily data\n", daily_dataframe)


Coordinates: 55.95916748046875°N -3.20684814453125°E
Elevation: 74.0 m asl
Timezone: b'Europe/London'b'GMT+1'
Timezone difference to GMT+0: 3600s

Daily data
                         date  weather_code  temperature_2m_max  \
0  2026-04-13 00:00:00+01:00           NaN                 NaN   
1  2026-04-14 00:00:00+01:00           NaN                 NaN   
2  2026-04-15 00:00:00+01:00           NaN                 NaN   
3  2026-04-16 00:00:00+01:00           NaN                 NaN   
4  2026-04-17 00:00:00+01:00           NaN                 NaN   
..                       ...           ...                 ...   
94 2026-07-16 00:00:00+01:00           3.0           18.094374   
95 2026-07-17 00:00:00+01:00           3.0           18.575499   
96 2026-07-18 00:00:00+01:00           3.0           21.575499   
97 2026-07-19 00:00:00+01:00           3.0           21.675499   
98 2026-07-20 00:00:00+01:00           2.0           25.025499   

    temperature_2m_min  wind_gusts_10m_max  rain

In [13]:
#Load historical csv of if i've worn a jumper
history = pd.read_csv("jumper_data.csv")

history["date"] = pd.to_datetime(
    history["date"],
    dayfirst=True
)

# Change Y/N to number
history["wear_jumper"] = history["wear_jumper"].map({
    "Y":1,
    "N":0
})


In [14]:
#Merge the data
#remove the timezone on the API data 
daily_dataframe["date"] = daily_dataframe["date"].dt.tz_localize(None)

training_data = pd.merge(
    daily_dataframe,
    history,
    on="date",
    how="inner"
)

#Define the features of the model
FEATURES = [
    "temperature_2m_max",
    "temperature_2m_min",
    "wind_gusts_10m_max",
    "rain_sum"
]

X = training_data[FEATURES]
y = training_data["wear_jumper"]

#Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)
#Try a logistic regression model
log_model = LogisticRegression(random_state=42)

log_model.fit(X_train, y_train)

log_predictions = log_model.predict(X_test)

print("===== Logistic Regression =====")
print("Accuracy:", accuracy_score(y_test, log_predictions))
print(classification_report(y_test, log_predictions))

#Try the random forest model
rf_model = RandomForestClassifier(
    random_state=42,
    n_estimators=100
)

rf_model.fit(X_train, y_train)

rf_predictions = rf_model.predict(X_test)

print("===== Random Forest =====")
print("Accuracy:", accuracy_score(y_test, rf_predictions))
print(classification_report(y_test, rf_predictions))


===== Logistic Regression =====
Accuracy: 0.8
              precision    recall  f1-score   support

           0       1.00      0.75      0.86         8
           1       0.50      1.00      0.67         2

    accuracy                           0.80        10
   macro avg       0.75      0.88      0.76        10
weighted avg       0.90      0.80      0.82        10

===== Random Forest =====
Accuracy: 0.8
              precision    recall  f1-score   support

           0       1.00      0.75      0.86         8
           1       0.50      1.00      0.67         2

    accuracy                           0.80        10
   macro avg       0.75      0.88      0.76        10
weighted avg       0.90      0.80      0.82        10



In [15]:
#Compare the models
log_cv = cross_val_score(
    log_model,
    X,
    y,
    cv=5,
    scoring="accuracy"
)

rf_cv = cross_val_score(
    rf_model,
    X,
    y,
    cv=5,
    scoring="accuracy"
)

print("\nCross-validation accuracy")
print("-------------------------")
print(f"Logistic Regression: {log_cv.mean():.3f}")
print(f"Random Forest:       {rf_cv.mean():.3f}")

#Pick the best model
if log_cv.mean() >= rf_cv.mean():
    model = log_model
    model_name = "Logistic Regression"
else:
    model = rf_model
    model_name = "Random Forest"

print(f"\nUsing {model_name} for forecasting.")

#If RF model look at the feature importance
if model_name == "Random Forest":

    importance = pd.Series(
        model.feature_importances_,
        index=FEATURES
    )

    importance.sort_values().plot.barh(figsize=(6,4))

    plt.title("Random Forest Feature Importance")
    plt.show()


Cross-validation accuracy
-------------------------
Logistic Regression: 0.820
Random Forest:       0.760

Using Logistic Regression for forecasting.


In [16]:
# Forecast using the best model
forecast = daily_dataframe.tail(7).copy()

forecast["prediction"] = model.predict(
    forecast[FEATURES]
)

forecast["recommendation"] = forecast["prediction"].map({
    1: "🧥 Wear a jumper",
    0: "😎 No jumper"
})

print(
    forecast[
        [
            "date",
            "temperature_2m_max",
            "temperature_2m_min",
            "wind_gusts_10m_max",
            "rain_sum",
            "recommendation"
        ]
    ]
)

         date  temperature_2m_max  temperature_2m_min  wind_gusts_10m_max  \
92 2026-07-14           15.651000           12.601000           34.919998   
93 2026-07-15           17.401001           12.901000           29.879999   
94 2026-07-16           18.094374           13.101000           26.639999   
95 2026-07-17           18.575499           13.275500           32.399998   
96 2026-07-18           21.575499           12.875501           32.760002   
97 2026-07-19           21.675499           12.725500           23.759998   
98 2026-07-20           25.025499           14.125501           30.239998   

    rain_sum   recommendation  
92       0.0  🧥 Wear a jumper  
93       0.0  🧥 Wear a jumper  
94       0.0      😎 No jumper  
95       0.0      😎 No jumper  
96       0.0      😎 No jumper  
97       0.0      😎 No jumper  
98       0.0      😎 No jumper  
